In [3]:
#!/usr/bin/env python3
import numpy as np
from itertools import combinations

def generate_dataset(n_reviewers=30, n_papers=20, lambda_=3, mu_=5):
    """
    Generates a synthetic matrix simulating reviewer-to-paper similarity scores.
    """
    np.random.seed(98)
    S = np.random.uniform(0.05, 0.95, (n_reviewers, n_papers))

    # Construct bottleneck test case
    S[:, 0] = np.random.uniform(0.05, 0.2, n_reviewers)
    S[0, 0] = 0.90
    S[1, 0] = 0.85

    return S

def evaluate_metrics(S, assignment):
    """
    Computes performance benchmarks.
    """
    m = S.shape[1]
    valuations = [sum(S[i, j] for i in assignment[j]) for j in range(m)]

    max_min_fairness = min(valuations)
    nsw = np.prod([max(v, 1e-5) for v in valuations]) ** (1.0 / m)

    return max_min_fairness, nsw

def run_brute_force_nsw(S, lambda_, mu_):
    """
    Explores all valid permutations of reviewer assignments using Depth First Search (Backtracking).
    It prunes branches where reviewer loads exceed mu_ to save time, and tracks the
    global maximum continuous Nash Social Welfare.
    """
    n_reviewers, n_papers = S.shape

    # Generate all possible combinations of lambda_ reviewers for a single paper
    all_combos = list(combinations(range(n_reviewers), lambda_))

    best_nsw = -1.0
    best_assignment = None

    def backtrack(paper_idx, current_assignment, reviewer_loads):
        nonlocal best_nsw, best_assignment

        # Base Case: All papers have been successfully assigned
        if paper_idx == n_papers:
            # Calculate metrics for this complete valid assignment
            _, current_nsw = evaluate_metrics(S, current_assignment)

            # Update global maximums if this permutation is better
            if current_nsw > best_nsw:
                best_nsw = current_nsw
                # Deep copy the current assignment
                best_assignment = {j: list(current_assignment[j]) for j in range(n_papers)}
            return

        # Recursive Case: Try all valid combinations for the current paper
        for combo in all_combos:
            # Pruning check: Can all reviewers in this combo take on another paper?
            if all(reviewer_loads[r] < mu_ for r in combo):

                # Apply assignment: update loads and path
                for r in combo:
                    reviewer_loads[r] += 1
                current_assignment.append(combo)

                # Recurse to the next paper
                backtrack(paper_idx + 1, current_assignment, reviewer_loads)

                # Backtrack: undo assignment to explore the next combo
                current_assignment.pop()
                for r in combo:
                    reviewer_loads[r] -= 1

    # Start the recursive search
    print(f"Starting brute force over {len(all_combos)} combinations per paper...")
    backtrack(0, [], [0] * n_reviewers)

    return best_assignment

# ==========================================
# EXECUTION & EXPERIMENTATION BLOCK
# ==========================================
if __name__ == "__main__":
    # --- SCALED DOWN CONFIGURATION FOR BRUTE FORCE ---
    # Warning: Keep these numbers small (e.g., < 10) or the combinatorial explosion will hang your PC.
    N_REVIEWERS = 6          # Size of reviewer pool
    N_PAPERS = 4             # Number of papers submitted
    DEMAND_PER_PAPER = 2     # Reviewers required per paper (lambda)
    LOAD_PER_REVIEWER = 2    # Max reviews per reviewer (mu)

    # Step 1: Generate Dataset
    print(f"Generating matrix for {N_REVIEWERS} Reviewers x {N_PAPERS} Papers...")
    S = generate_dataset(n_reviewers=N_REVIEWERS, n_papers=N_PAPERS,
                         lambda_=DEMAND_PER_PAPER, mu_=LOAD_PER_REVIEWER)

    # Step 2: Run Brute Force
    print("\nRunning Brute Force Optimization (Continuous NSW Maximization)...")
    print("Please wait, computing permutations...")

    bf_assignment = run_brute_force_nsw(S, lambda_=DEMAND_PER_PAPER, mu_=LOAD_PER_REVIEWER)

    if bf_assignment is None:
        print("\nNo valid assignment could be found that satisfies both demand and capacity constraints.")
    else:
        # Step 3: Compute Metrics
        bf_fairness, bf_nsw = evaluate_metrics(S, bf_assignment)

        # Step 4: Format and Print Results Panel
        print("\n" + "="*60)
        print("                   EXPERIMENT RESULTS")
        print("="*60)
        print(f"{'Algorithm':<20} | {'Min Fairness':<15} | {'NSW (Geom Mean)':<15}")
        print("-" * 60)
        print(f"{'Brute Force Optimal':<20} | {bf_fairness:<15.4f} | {bf_nsw:<15.4f}")
        print("="*60)

        print("\nOptimal Assignment Mapping:")
        for paper, reviewers in bf_assignment.items():
            print(f"Paper {paper}: Reviewers {reviewers}")

Generating matrix for 6 Reviewers x 4 Papers...

Running Brute Force Optimization (Continuous NSW Maximization)...
Please wait, computing permutations...
Starting brute force over 15 combinations per paper...

                   EXPERIMENT RESULTS
Algorithm            | Min Fairness    | NSW (Geom Mean)
------------------------------------------------------------
Brute Force Optimal  | 1.2379          | 1.5936         

Optimal Assignment Mapping:
Paper 0: Reviewers [0, 1]
Paper 1: Reviewers [0, 4]
Paper 2: Reviewers [4, 5]
Paper 3: Reviewers [3, 5]


#### Algorithm       | Min Fairness (Max-Min) | Nash Social Welfare (NSW)
----------------------------------------------------------------------
#### PR4A            | 1.2379                 | 1.5936                   
#### Binary NSW      | 1.2379                 | 1.5106
==================================================